Baseline Pipeline

In [1]:
import time
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

In [15]:
def run_baseline_pipeline(data_path):
    print(f"\n==========================================")
    print(f"RUNNING BASELINE PIPELINE ON: {data_path}")
    print(f"==========================================")
    
    # 1. Load Dataset
    df = pd.read_csv(data_path)
    
    # 2. Filter to Relevant Fraud Transaction Types
    df_filtered = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
    
    # 3. Engineer Relational Balance Feature
    df_filtered['balance_diff_orig'] = (
        df_filtered['oldbalanceOrg'] - df_filtered['amount'] - df_filtered['newbalanceOrig']
    )
    
    # 4. Define Feature Space & Naive Impure Data Handling (fillna 0 for baseline)
    feature_cols = ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'balance_diff_orig']
    X_baseline = df_filtered[feature_cols].fillna(0)
    y_baseline = df_filtered['isFraud']
    
    # 5. Determine Contamination Rate
    contamination_rate = y_baseline.mean()
    
    # 6. Time Model Execution (Latency Delta T)
    start_time = time.time()
    
    model = IsolationForest(
        n_estimators=100,
        contamination=contamination_rate,
        random_state=42,
        n_jobs=-1
    )
    
    preds = model.fit_predict(X_baseline)
    end_time = time.time()
    
    # 7. Map Output: Isolation Forest -1 (Anomaly) -> 1, 1 (Normal) -> 0
    preds_binary = (preds == -1).astype(int)
    
    # 8. Compute Performance & Timing Metrics
    delta_t = end_time - start_time
    precision = precision_score(y_baseline, preds_binary, pos_label=1, zero_division=0)
    recall = recall_score(y_baseline, preds_binary, pos_label=1, zero_division=0)
    f1 = f1_score(y_baseline, preds_binary, pos_label=1, zero_division=0)
    cm = confusion_matrix(y_baseline, preds_binary)
    
    # 9. Print Results
    print(f"Execution Latency (Delta T): {delta_t:.4f} seconds")
    print(f"Confusion Matrix:\n{cm}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    
    return {
        "data_path": data_path,
        "delta_t": delta_t,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

# Execute on Pristine Data
if __name__ == "__main__":
    baseline_results = run_baseline_pipeline('Dataset.csv')


RUNNING BASELINE PIPELINE ON: Dataset.csv
Execution Latency (Delta T): 47.9126 seconds
Confusion Matrix:
[[2754148    8048]
 [   8048     165]]
Precision: 0.0201
Recall: 0.0201
F1-Score: 0.0201


In [16]:
run_baseline_pipeline('paysim_noise_5.csv')
run_baseline_pipeline('paysim_noise_10.csv')
run_baseline_pipeline('paysim_noise_20.csv')


RUNNING BASELINE PIPELINE ON: paysim_noise_5.csv
Execution Latency (Delta T): 43.8161 seconds
Confusion Matrix:
[[2754279    7917]
 [   7917     296]]
Precision: 0.0360
Recall: 0.0360
F1-Score: 0.0360

RUNNING BASELINE PIPELINE ON: paysim_noise_10.csv
Execution Latency (Delta T): 42.8620 seconds
Confusion Matrix:
[[2754349    7847]
 [   7847     366]]
Precision: 0.0446
Recall: 0.0446
F1-Score: 0.0446

RUNNING BASELINE PIPELINE ON: paysim_noise_20.csv
Execution Latency (Delta T): 40.7093 seconds
Confusion Matrix:
[[2754255    7941]
 [   7945     268]]
Precision: 0.0326
Recall: 0.0326
F1-Score: 0.0326


{'data_path': 'paysim_noise_20.csv',
 'delta_t': 40.709258794784546,
 'precision': 0.03264709465221099,
 'recall': 0.03263119444782662,
 'f1_score': 0.032639142613567165}